In [14]:
import os, glob, ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.path import Path
from matplotlib.markers import MarkerStyle
from scipy.signal import find_peaks

# -----------------------
# Teardrop marker
# -----------------------
verts = [
    (0.0, 1.0), (-0.5, 0.2), (-0.3, -0.6), (0.0, -1.0),
    (0.3, -0.6), (0.5, 0.2), (0.0, 1.0)
]
codes = [Path.MOVETO] + [Path.LINETO] * (len(verts) - 2) + [Path.CLOSEPOLY]
teardrop_marker = MarkerStyle(Path(verts, codes))

# -----------------------
# Inputs/Outputs
# -----------------------
phenotypes = {
    "No_Invasion": "No_Invasion.png",
    "Single_cell_invasion": "Single_Cell_Invasion.png",
    "Bulk_Invasion": "Bulk_Invasion.png",
    "Multimodal_invasion": "Multimodal_Invasion.png"
}

results_base = "../../Results/Sample"
save_base = "../../Results/Plots/Phenotype_Composite"
os.makedirs(save_base, exist_ok=True)

# -----------------------
# Helpers
# -----------------------
def newest(path_pattern: str) -> str | None:
    """Pick the newest file matching a glob; return None if none."""
    matches = glob.glob(path_pattern)
    if not matches:
        return None
    matches.sort(key=os.path.getmtime, reverse=True)
    return matches[0]

def require_all(paths: list[str | None]) -> bool:
    return all(p is not None and os.path.exists(p) for p in paths)

def safe_eval_list_series(series: pd.Series) -> list[list[float]]:
    """ast.literal_eval each cell, tolerating NaN/'' -> []."""
    out = []
    for v in series.fillna("[]"):
        try:
            parsed = ast.literal_eval(v) if isinstance(v, str) else []
        except Exception:
            parsed = []
        out.append(parsed if isinstance(parsed, list) else [])
    return out

def label_once(ax, label: str):
    """Return label only the first time we add it."""
    _, labs = ax.get_legend_handles_labels()
    return label if label not in labs else ""

# -----------------------
# Main plotting function
# -----------------------
def process_phenotype(phenotype_folder: str, output_filename: str):
    base_folder = os.path.join(results_base, phenotype_folder)
    # Pick newest run per file type
    boundary_file = newest(os.path.join(base_folder, 'PositionData_*', 'BoundaryData_*.csv'))
    defector_file = newest(os.path.join(base_folder, 'PositionData_*', 'DefectorPosition_*.csv'))
    tumor_file    = newest(os.path.join(base_folder, 'PositionData_*', 'TumorPosition_*.csv'))
    leader_file   = newest(os.path.join(base_folder, 'PositionData_*', 'TumorLeaderCells_*.csv'))
    follower_file = newest(os.path.join(base_folder, 'PositionData_*', 'TumorFollowerCells_*.csv'))
    cluster_file  = newest(os.path.join(base_folder, 'PositionData_*', 'ClusterComposition_*.csv'))

    needed = [boundary_file, defector_file, tumor_file, leader_file, follower_file, cluster_file]
    if not require_all(needed):
        print(f"Skipping {phenotype_folder}: Required files not found.")
        return

    # Load data
    boundary_df = pd.read_csv(boundary_file)
    defector_df = pd.read_csv(defector_file)
    tumor_df    = pd.read_csv(tumor_file)
    leader_df   = pd.read_csv(leader_file)
    follower_df = pd.read_csv(follower_file)
    cluster_df  = pd.read_csv(cluster_file)
    cluster_df.columns = cluster_df.columns.str.strip()

    # Column sanity for boundary_df (your file uses this exact spelling)
    # If your CSV sometimes spells it differently, add fallbacks here.
    required_boundary_cols = ["X", "Main_Tumor", "Outermost", "Lowesr_boundary_point"]
    for c in required_boundary_cols:
        if c not in boundary_df.columns:
            print(f"Skipping {phenotype_folder}: Missing boundary column '{c}'.")
            return

    x = boundary_df["X"].to_numpy()
    main_tumor = boundary_df["Main_Tumor"].to_numpy()
    outermost = boundary_df["Outermost"].to_numpy()
    lowest_point = boundary_df["Lowesr_boundary_point"].to_numpy()

    # Detect “fingers” on the core boundary
    raw_peaks, _ = find_peaks(main_tumor, prominence=10, distance=10, width=5)
    finger_peaks = []
    min_sep = 15
    last = -np.inf
    for p in raw_peaks:
        if p - last > min_sep:
            finger_peaks.append(p)
            last = p
    finger_peaks = np.array(finger_peaks, dtype=int)

    # Optional per-member coordinates inside clusters
    has_member_columns = all(col in cluster_df.columns for col in
                             ["Member_Leader_X", "Member_Leader_Y", "Member_Follower_X", "Member_Follower_Y"])
    if has_member_columns:
        lxs = safe_eval_list_series(cluster_df["Member_Leader_X"])
        lys = safe_eval_list_series(cluster_df["Member_Leader_Y"])
        fxs = safe_eval_list_series(cluster_df["Member_Follower_X"])
        fys = safe_eval_list_series(cluster_df["Member_Follower_Y"])
        cluster_leader_xs = [x for sub in lxs for x in sub]
        cluster_leader_ys = [y for sub in lys for y in sub]
        cluster_follower_xs = [x for sub in fxs for x in sub]
        cluster_follower_ys = [y for sub in fys for y in sub]

    # Figure size rules (your “format above”)
    wide = ("Single_Cell_Invasion" in output_filename) or ("Multimodal" in output_filename)
    figsize = (12, 7) if wide else (8, 2)
    fig, ax = plt.subplots(figsize=figsize)

    # Boundaries
    ax.plot(x, main_tumor, label="Core Tumor Boundary", linewidth=10, color="red")
    ax.plot(x, outermost, label="Outermost Boundary", linewidth=2, color="orange")
    ax.scatter(x, outermost, color="white", s=10)
    ax.plot(x, lowest_point, label="Lowest Point", linestyle="--", linewidth=10, color="purple")

    # Cell-level annotations
    ax.scatter(follower_df["x"], follower_df["y"], color="green", marker="o", s=120,
               label=label_once(ax, "Follower"))
    ax.quiver(
        leader_df["x"], leader_df["y"],
        np.zeros_like(leader_df["x"]), np.full_like(leader_df["y"], 4),
        color="blue", angles="xy", scale_units="xy", scale=0.5, width=0.005,
        label=label_once(ax, "Leader (direction)")
    )
    ax.scatter(defector_df["x"], defector_df["y"], color="deepskyblue", marker="*", s=300,
               label=label_once(ax, "Defector"))
    ax.scatter(cluster_df["Centroid_X"], cluster_df["Centroid_Y"], color="magenta", marker="X", s=300,
               label=label_once(ax, "Cluster Centroid"))

    # Finger peaks for bulk/multimodal
    if ("Bulk" in output_filename) or ("Multimodal" in output_filename):
        if finger_peaks.size:
            ax.scatter(x[finger_peaks], main_tumor[finger_peaks], marker="P", color="gold", s=150,
                       label=label_once(ax, "Boundary Peak"))

    # Show cluster members (multimodal only)
    if ("Multimodal" in output_filename) and has_member_columns:
        if cluster_leader_xs:
            ax.scatter(cluster_leader_xs, cluster_leader_ys, marker=teardrop_marker, s=300, color="blue",
                       label=label_once(ax, "Leader in Cluster"))
        if cluster_follower_xs:
            ax.scatter(cluster_follower_xs, cluster_follower_ys, color="lime", s=200,
                       label=label_once(ax, "Follower in Cluster"))

    # Minimal axes look
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(""); ax.set_ylabel("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(colors="white")

    # Axis limits (your rules)
    if ("Bulk" in output_filename) or ("No_Invasion" in output_filename):
        ax.set_xlim(0, 250); ax.set_ylim(0, 80)
    elif "Single_Cell_Invasion" in output_filename:
        ax.set_xlim(0, 250)
    else:
        ax.set_xlim(83, 380)

    # Build a compact legend (only if anything labeled survived)
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        ax.legend(handles, labels, loc="upper right", frameon=False, fontsize=10)

    fig.tight_layout()
    outpath = os.path.join(save_base, output_filename)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {outpath}")

# -----------------------
# Run all phenotypes
# -----------------------
for folder, filename in phenotypes.items():
    process_phenotype(folder, filename)


Saved: ../../Results/Plots/Phenotype_Composite/No_Invasion.png
Saved: ../../Results/Plots/Phenotype_Composite/Single_Cell_Invasion.png
Saved: ../../Results/Plots/Phenotype_Composite/Bulk_Invasion.png
Saved: ../../Results/Plots/Phenotype_Composite/Multimodal_Invasion.png


In [36]:
from pathlib import Path
from PIL import Image
import os

# ================================
# Config
# ================================
snap_dir = Path("/Users/sheriffakeeb/Desktop/TumorInvasion/Results/scan_8455_Jlf_1_mu_30_PP_0.9_rep_5/Tumor_Snapshots")
out_dir  = Path("/Users/sheriffakeeb/Desktop/TumorInvasion/Results/scan_8455_Jlf_1_mu_30_PP_0.9_rep_5/Split_Snapshots")

NUM_SLICES = 10                 # vertical slices per frame
KEEP_BOTTOM_FRACTION = 0.5      # keep bottom half; e.g., 0.4 keeps bottom 40%
GIF_FRAME_DURATION_MS = 120     # per-frame duration in GIF
GIF_LOOP = 0                    # 0 = loop forever
VIDEO_FPS = 8                   # FPS for MP4 (if imageio/ffmpeg available)

out_dir.mkdir(parents=True, exist_ok=True)

# Slice subfolders
slice_dirs = {}
for i in range(1, NUM_SLICES + 1):
    sd = out_dir / f"slice_{i}"
    sd.mkdir(parents=True, exist_ok=True)
    slice_dirs[i] = sd

# Animation folders
anim_root = out_dir / "Animations"
gif_dir = anim_root / "GIFs"
mp4_dir = anim_root / "MP4s"
gif_dir.mkdir(parents=True, exist_ok=True)
mp4_dir.mkdir(parents=True, exist_ok=True)

# Track saved images per slice for animation building
slice_image_paths = {i: [] for i in range(1, NUM_SLICES + 1)}

# ================================
# Process snapshots
# ================================
files = sorted(snap_dir.glob("Tumor_Snapshots_*.png"))

for f in files:
    img = Image.open(f)
    w, h = img.size
    slice_w = w // NUM_SLICES

    # Height cropping parameters for bottom portion
    keep_top = int(h * (1 - KEEP_BOTTOM_FRACTION))  # start row of bottom region
    keep_bottom = h

    for i in range(NUM_SLICES):
        left = i * slice_w
        right = (i + 1) * slice_w if i < NUM_SLICES - 1 else w  # last slice takes remainder

        # 1) Crop to vertical slice
        slice_img = img.crop((left, 0, right, h))
        # 2) Crop to bottom fraction
        final_crop = slice_img.crop((0, keep_top, slice_img.width, keep_bottom))

        # Save per-slice still image
        out_name = slice_dirs[i+1] / f"{f.stem}_slice_{i+1}.png"
        final_crop.save(out_name)
        slice_image_paths[i+1].append(out_name)

print(f"✅ Slices (bottom {int(KEEP_BOTTOM_FRACTION*100)}%) saved under: {out_dir}")

# ================================
# Build per-slice GIFs
# ================================
for i in range(1, NUM_SLICES + 1):
    frames = slice_image_paths[i]
    if not frames:
        continue

    # Load with PIL
    pil_frames = [Image.open(p) for p in frames]
    gif_path = gif_dir / f"slice_{i}.gif"
    # Save animated GIF
    pil_frames[0].save(
        gif_path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=GIF_FRAME_DURATION_MS,
        loop=GIF_LOOP,
        optimize=False,
        disposal=2  # helps some viewers render frame-by-frame correctly
    )

print(f"✅ All animations (GIFs/MP4s) are under: {anim_root}")


✅ Slices (bottom 50%) saved under: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/scan_8455_Jlf_1_mu_30_PP_0.9_rep_5/Split_Snapshots
✅ All animations (GIFs/MP4s) are under: /Users/sheriffakeeb/Desktop/TumorInvasion/Results/scan_8455_Jlf_1_mu_30_PP_0.9_rep_5/Split_Snapshots/Animations
